# TH - axios npm Supply Chain (TeamPCP)

Hunting for indicators of the compromised `axios` npm packages across customer environments. The attacker hijacked the `jasonsaayman` maintainer account and published `axios@1.14.1` and `axios@0.30.4` with a phantom dependency (`plain-crypto-js@4.2.1`) whose postinstall hook drops a cross-platform RAT via `sfrclak.com:8000`.

**Date:** 2026-03-31    
**Author:** Josh Strickland  
**Platform:** LimaCharlie EDR  
**Advisory:** [StepSecurity -- axios Compromised on npm](https://www.stepsecurity.io/blog/axios-compromised-on-npm-malicious-versions-drop-remote-access-trojan)  
**Exposure window:** 2026-03-31 00:21 to ~03:15 UTC (~2h 53m for 1.14.1, ~2h 15m for 0.30.4)  
**Target OS:** macOS, Windows, Linux (cross-platform RAT dropper)  
**Attribution:** TeamPCP (same actor as LiteLLM PyPI compromise, 2026-03-24)  

## Hypothesis

Any system that ran `npm install` resolving `axios@1.14.1` or `axios@0.30.4` between 2026-03-31 00:21 UTC and ~03:15 UTC executed a cross-platform RAT dropper. The compromised maintainer account injected `plain-crypto-js@4.2.1` as a phantom dependency, never imported anywhere in axios source, exists solely to trigger a `postinstall` hook that drops a platform-specific stage-2 payload from `sfrclak.com:8000`.

The dropper (`setup.js`) is obfuscated with XOR cipher + base64. After execution it self-deletes and swaps its own `package.json` with a clean stub reporting version 4.2.0. Post-incident inspection of `node_modules/plain-crypto-js/package.json` shows no trace of the postinstall trigger.

Same threat actor (TeamPCP) behind the LiteLLM PyPI compromise 3 days earlier. Proton Mail accounts, single-purpose C2 domains, anti-forensics cleanup.

Refs: [StepSecurity](https://www.stepsecurity.io/blog/axios-compromised-on-npm-malicious-versions-drop-remote-access-trojan), [npm advisory](https://www.npmjs.com/package/axios), [axios #10604](https://github.com/axios/axios/issues/10604)

## Methodology

Two-phase hunt across all organizations, then per-host correlation. Validated against a controlled detonation before sweeping production.

**Phase 1 -- Wide sweep (all orgs, all platforms):** 5 queries targeting C2 DNS resolution, C2 TCP connections, postinstall dropper execution, curl fetching the stage-2 payload, and stage-2 file drops via NEW_DOCUMENT. Near-zero false positive rates because they target specific attacker infrastructure and campaign artifacts.

**Phase 2 -- Platform-specific artifacts (hit orgs from Phase 1):** 6 queries targeting platform-specific RAT delivery -- macOS osascript dropper and RAT binary at `/Library/Caches/com.apple.act.mond`, Windows PowerShell copy at `%PROGRAMDATA%\wt.exe`, campaign-specific VBS/PS1 loaders, and Linux `/tmp/ld.py` execution.

**Phase 3 -- Per-host correlation:** Merge both phases by hostname to build an attack stage matrix.

### Setup

In [ ]:
# Install requirements into the running kernel (no restart needed)
# Pin limacharlie to v4.x -- v5 removed Manager/Replay classes
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "limacharlie", "-y"], stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "limacharlie<5", "pandas", "python-dotenv", "--quiet", "--no-cache-dir"])
print("Requirements installed")

In [ ]:
import limacharlie
from limacharlie.Replay import Replay
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import json
import time
import os
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

# Show all columns and full cell content in notebook DataFrame renders
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 200)

print("Imports loaded, .env processed")

### Credentials

In [ ]:
LC_UID = os.environ["LC_UID"]
LC_API_KEY = os.environ["LC_API_KEY"]

assert LC_UID, "LC_UID not found in .env"
assert LC_API_KEY, "LC_API_KEY not found in .env"
print(f"UID: {LC_UID[:8]}...")
print("Credentials loaded")

### Discover Organizations

In [ ]:
mgr = limacharlie.Manager(uid=LC_UID, secret_api_key=LC_API_KEY)
result = mgr.userAccessibleOrgs(with_names=True, limit=500)
orgs = dict(zip(result['orgs'], [result['names'].get(oid, 'Unknown') for oid in result['orgs']]))

print(f"Found {len(orgs)} organizations:")
for oid, name in sorted(orgs.items(), key=lambda x: x[1]):
    print(f"  {name}: {oid[:12]}...")

### Filter Orgs (optional)

In [ ]:
# --- Uncomment to exclude specific orgs ---
# EXCLUDE_ORGS = {"example1", "example2"}
# orgs = {oid: name for oid, name in orgs.items() if name not in EXCLUDE_ORGS}

# --- Uncomment to target only specific orgs ---
# INCLUDE_ONLY = {"customer-a", "customer-b"}
# orgs = {oid: name for oid, name in orgs.items() if name in INCLUDE_ONLY}

print(f"Querying {len(orgs)} org(s)")

### Shared Query Function + Tuning Parameters

In [ ]:
LIMIT_PER_ORG = 5000
LIMIT_EVENT_SCAN = 200000   
MAX_WORKERS = 15


def expand_events(df):
    """Expand event_data JSON into evt_ prefixed columns. Returns new DataFrame."""
    if df.empty or 'event_data' not in df.columns:
        return df
    expanded = pd.json_normalize(df['event_data'].apply(json.loads))
    expanded.columns = ['evt_' + c for c in expanded.columns]
    return pd.concat([df.drop(columns=['event_data']), expanded], axis=1)


def query_single_org(oid, org_name, query_name, lcql_query):
    """Run one LCQL query against one org. Returns (events_list, error_or_None)."""
    try:
        mgr = limacharlie.Manager(oid=oid, uid=LC_UID, secret_api_key=LC_API_KEY)
        replay = Replay(mgr)

        response = replay._doQuery(
            lcql_query,
            limitEvent=LIMIT_EVENT_SCAN,
            limitEval=None,
            isCursorBased=False,
            stream='event'
        )

        error = response.get('error', '')
        results = response.get('results', [])

        if error and not results and 'limit' not in error.lower():
            return [], f"Error: {error}"

        events = []
        for result in results:
            evt = result.get('data', result) if isinstance(result, dict) else None
            if not evt:
                continue

            routing = evt.get('routing', {})
            hostname = routing.get('hostname', '')

            if hostname == 'ext-cloud-cli':
                continue

            ts = routing.get('event_time', 0)
            if ts > 9999999999:
                ts = ts / 1000

            row = {
                'query_name': query_name,
                'org_name': org_name,
                'oid': oid,
                'timestamp': datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if ts else '',
                'event_type': routing.get('event_type', ''),
                'hostname': hostname,
                'sensor_id': routing.get('sid', ''),
                'event_data': json.dumps(evt.get('event', {}), default=str),
            }
            events.append(row)

        events.sort(key=lambda e: e['timestamp'], reverse=True)
        return events[:LIMIT_PER_ORG], None

    except Exception as e:
        return [], str(e)


def run_queries(queries, target_orgs, label=""):
    """Run a dict of {name: lcql} queries across target_orgs. Returns (all_events, errors)."""
    all_events = []
    errors = []
    start = time.time()

    total = len(queries) * len(target_orgs)
    print(f"{label}: {len(queries)} queries x {len(target_orgs)} orgs = {total} jobs")

    for query_name, lcql_query in queries.items():
        print(f"\n  --- {query_name} ---")

        workers = min(MAX_WORKERS, len(target_orgs))
        query_events = []

        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(query_single_org, oid, name, query_name, lcql_query): (oid, name)
                for oid, name in target_orgs.items()
            }

            for future in as_completed(futures):
                oid, name = futures[future]
                events, error = future.result()

                if error:
                    errors.append({'query': query_name, 'org': name, 'oid': oid, 'error': error})
                elif events:
                    query_events.extend(events)
                    print(f"      HIT: {name} -- {len(events)} events")

        if not query_events:
            print(f"      No hits")
        all_events.extend(query_events)

    elapsed = time.time() - start
    print(f"\n  {label} complete: {len(all_events)} events, {len(errors)} errors, {elapsed:.1f}s")
    return all_events, errors

print("Query function ready")

### Phase 1 -- Wide Sweep

C2 communication + dropper execution across all orgs and platforms. The compromised packages were live for ~3 hours, but cached installs and lockfile-pinned versions could trigger later.

In [ ]:
PHASE1_QUERIES = {
    # IOC: DNS resolution of attacker C2 domain
    "ioc_c2_dns": (
        "-336h | * | DNS_REQUEST | "
        "event/DOMAIN_NAME contains 'sfrclak.com'"
    ),

    # IOC: TCP connection to C2 IP (catches beaconing even if DNS is cached,
    # resolved via /etc/hosts, or bypassed entirely with IP literal)
    "ioc_c2_tcp": (
        "-336h | * | NEW_TCP4_CONNECTION | "
        "event/DESTINATION contains '142.11.206.73'"
    ),

    # Behavioral: npm postinstall hook executing the dropper
    # setup.js is the decoded XOR+base64 obfuscated dropper from plain-crypto-js
    "behavior_postinstall_exec": (
        "-336h | * | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'node setup.js'"
    ),

    # IOC: curl downloading stage-2 payload from the C2
    # All three platform branches use curl to fetch the payload
    "ioc_c2_curl_fetch": (
        "-336h | * | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'curl' "
        "and event/COMMAND_LINE contains 'sfrclak'"
    ),

    # IOC: Stage-2 file drops (NEW_DOCUMENT fires on default telemetry)
    # Catches the payload hitting disk regardless of delivery method
    "ioc_stage2_drop": (
        "-336h | * | NEW_DOCUMENT | "
        "event/FILE_PATH contains 'com.apple.act.mond' "
        "or event/FILE_PATH contains '/tmp/ld.py' "
        "or event/FILE_PATH contains '6202033'"
    ),
}

phase1_events, phase1_errors = run_queries(PHASE1_QUERIES, orgs, label="PHASE 1: WIDE SWEEP")

### Phase 1 Results

Pull out which orgs and hosts had hits. These become the targets for Phase 2.

In [ ]:
df_phase1 = pd.DataFrame(phase1_events)

if not df_phase1.empty:
    df_phase1['timestamp'] = pd.to_datetime(df_phase1['timestamp'], utc=True)
    df_phase1 = df_phase1.sort_values('timestamp', ascending=False).reset_index(drop=True)

    # Extract hit orgs and hosts for Phase 2 targeting
    hit_oids = set(df_phase1['oid'].unique())
    hit_orgs = {oid: name for oid, name in orgs.items() if oid in hit_oids}
    hit_hosts = set(df_phase1['hostname'].unique())

    # Expand event JSON into columns
    df_phase1_expanded = expand_events(df_phase1)

    print(f"PHASE 1 HITS: {len(df_phase1_expanded)} events across {len(hit_orgs)} org(s), {len(hit_hosts)} host(s)")
    print(f"Columns: {len(df_phase1_expanded.columns)}")
    print()
    print("Hit breakdown by query:")
    print(df_phase1_expanded['query_name'].value_counts().to_string())
    print()
    display(df_phase1_expanded)
else:
    df_phase1_expanded = pd.DataFrame()
    hit_orgs = {}
    hit_hosts = set()
    print("PHASE 1: No hits. Running Phase 2 across ALL orgs as a broader check.")

### Phase 2 -- Platform-Specific Artifacts

Targeted queries for the RAT delivery indicators on each OS. Runs against hit orgs from Phase 1, or all orgs if Phase 1 was clean.

In [ ]:
PHASE2_QUERIES = {
    # macOS: osascript executing the campaign-specific AppleScript dropper from /tmp
    # This is the intermediate step that writes the RAT binary to /Library/Caches
    "behavior_macos_osascript_dropper": (
        "-336h | plat == macos | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'osascript' "
        "and event/COMMAND_LINE contains '6202033'"
    ),

    # macOS: RAT binary disguised as Apple daemon in /Library/Caches
    "ioc_macos_rat_binary": (
        "-336h | plat == macos | NEW_PROCESS | "
        "event/FILE_PATH contains 'com.apple.act.mond'"
    ),

    # Windows: Renamed PowerShell copy in ProgramData (persistent artifact)
    # wt.exe persists across reinstalls -- only created if not already present
    "ioc_windows_ps_masquerade": (
        "-336h | plat == windows | NEW_PROCESS | "
        "event/FILE_PATH contains 'ProgramData' "
        "and event/FILE_PATH contains 'wt.exe'"
    ),

    # Windows: Campaign-specific VBScript and PowerShell loaders in temp
    "ioc_windows_campaign_loader": (
        "-336h | plat == windows | NEW_PROCESS | "
        "event/COMMAND_LINE contains '6202033.vbs' "
        "or event/COMMAND_LINE contains '6202033.ps1'"
    ),

    # Linux: Python stage-2 RAT executing from /tmp
    "behavior_linux_tmp_python": (
        "-336h | plat == linux | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'python3 /tmp/ld.py'"
    ),

    # Linux: curl writing stage-2 to /tmp/ld.py
    "behavior_linux_curl_dropper": (
        "-336h | plat == linux | NEW_PROCESS | "
        "event/COMMAND_LINE contains 'curl' "
        "and event/COMMAND_LINE contains '/tmp/ld.py'"
    ),
}

# Target only hit orgs from Phase 1, or all orgs if Phase 1 was clean
phase2_target = hit_orgs if hit_orgs else orgs
phase2_events, phase2_errors = run_queries(PHASE2_QUERIES, phase2_target, label="PHASE 2: PLATFORM ARTIFACTS")

### Phase 2 Results

In [ ]:
df_phase2 = pd.DataFrame(phase2_events)

if not df_phase2.empty:
    df_phase2['timestamp'] = pd.to_datetime(df_phase2['timestamp'], utc=True)
    df_phase2 = df_phase2.sort_values('timestamp', ascending=False).reset_index(drop=True)

    # Expand event JSON into columns
    df_phase2_expanded = expand_events(df_phase2)

    print(f"PHASE 2 HITS: {len(df_phase2_expanded)} events across {df_phase2['oid'].nunique()} org(s), {df_phase2['hostname'].nunique()} host(s)")
    print(f"Columns: {len(df_phase2_expanded.columns)}")
    print()
    print("Hit breakdown by query:")
    print(df_phase2_expanded['query_name'].value_counts().to_string())
    print()
    display(df_phase2_expanded)
else:
    df_phase2_expanded = pd.DataFrame()
    print("PHASE 2: No hits.")

### Phase 3 -- Correlate

Merge both phases and build a per-host matrix showing which attack stages were observed. Multiple stages on the same host = high confidence.

In [ ]:
# Merge all events from both phases
all_events = phase1_events + phase2_events
all_errors = phase1_errors + phase2_errors

df = pd.DataFrame(all_events)

if not df.empty:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp', ascending=False).reset_index(drop=True)

    # Map query names to attack stages
    STAGE_MAP = {
        'ioc_c2_dns': 'c2_comms',
        'ioc_c2_tcp': 'c2_comms',
        'behavior_postinstall_exec': 'dropper',
        'ioc_c2_curl_fetch': 'c2_comms',
        'ioc_stage2_drop': 'stage2_drop',
        'behavior_macos_osascript_dropper': 'rat_delivery',
        'ioc_macos_rat_binary': 'rat_delivery',
        'ioc_windows_ps_masquerade': 'rat_delivery',
        'ioc_windows_campaign_loader': 'rat_delivery',
        'behavior_linux_tmp_python': 'rat_delivery',
        'behavior_linux_curl_dropper': 'rat_delivery',
    }

    df['attack_stage'] = df['query_name'].map(STAGE_MAP)

    # Build per-host attack stage matrix
    host_summary = []
    for (hostname, org_name), group in df.groupby(['hostname', 'org_name']):
        stages = set(group['attack_stage'].dropna().unique())
        first_seen = group['timestamp'].min()
        last_seen = group['timestamp'].max()
        sensor_id = group['sensor_id'].iloc[0]

        host_summary.append({
            'hostname': hostname,
            'org_name': org_name,
            'sensor_id': sensor_id,
            'dropper': 'dropper' in stages,
            'c2_comms': 'c2_comms' in stages,
            'stage2_drop': 'stage2_drop' in stages,
            'rat_delivery': 'rat_delivery' in stages,
            'stages_seen': len(stages),
            'total_events': len(group),
            'first_seen': first_seen,
            'last_seen': last_seen,
        })

    df_hosts = pd.DataFrame(host_summary).sort_values('stages_seen', ascending=False).reset_index(drop=True)

    # Confidence label
    df_hosts['confidence'] = df_hosts['stages_seen'].apply(
        lambda s: 'CONFIRMED' if s >= 3 else 'HIGH' if s >= 2 else 'INVESTIGATE'
    )

    # Expand event data for the full correlated dataset
    df_expanded = expand_events(df)

    print("=" * 70)
    print("HOST SUMMARY MATRIX")
    print("=" * 70)
    confirmed = len(df_hosts[df_hosts['stages_seen'] >= 3])
    high = len(df_hosts[df_hosts['stages_seen'] == 2])
    investigate = len(df_hosts[df_hosts['stages_seen'] == 1])
    print(f"{len(df_hosts)} host(s): {confirmed} CONFIRMED, {high} HIGH, {investigate} INVESTIGATE")
    print()
    display(df_hosts)

else:
    df_expanded = df
    df_hosts = pd.DataFrame()
    print("No hits across either phase. Environment appears clean.")

### All Events -- Expanded

Full correlated dataset from both phases with event JSON flattened into `evt_` columns. Same data exported to CSV but viewable inline.

In [ ]:
if not df_expanded.empty:
    print(f"All events: {len(df_expanded)} rows x {len(df_expanded.columns)} columns")
    print()
    display(df_expanded)
else:
    print("No events to display.")

### Export

Host summary matrix + full expanded event dataset to CSV for triage in DataWrangler.

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

if not df_hosts.empty:
    summary_file = f"axios_hunt_hosts_{ts}.csv"
    df_hosts.to_csv(summary_file, index=False)
    print(f"Host summary: {summary_file} ({len(df_hosts)} hosts)")

if not df_expanded.empty:
    events_file = f"axios_hunt_events_{ts}.csv"
    df_expanded.to_csv(events_file, index=False)
    print(f"Full events:  {events_file} ({len(df_expanded)} events, {len(df_expanded.columns)} columns)")

if df_hosts.empty and df_expanded.empty:
    print("Nothing to export -- no hits found")

### Errors

In [ ]:
if all_errors:
    print(f"{len(all_errors)} error(s):")
    for e in all_errors:
        print(f"  [{e['query']}] {e['org']}: {e['error']}")
else:
    print("No errors")

---

## Findings

**Result:** No customer impact. All hits from controlled sandbox detonation only.

Swept all orgs over 14 days. Zero hits in production environments. The only host flagged was the controlled sandbox, where both the dropper package (`plain-crypto-js-4.2.1.tgz`) and the Linux RAT (`ld.py` from MalwareBazaar) were detonated. Full kill chain confirmed in sandbox telemetry: postinstall trigger, curl C2 fetch, NEW_DOCUMENT file drop, RAT execution with TCP callbacks. Exposure window was ~3 hours on 2026-03-31; both malicious axios versions were unpublished by npm before any customer CI/CD pipelines ran morning builds.

### Recommendations

1. **Block IOCs.** Add `sfrclak[.]com` and `142.11.206[.]73` to DNS blocklists and firewall deny rules.
2. **Check lockfiles.** Any `package-lock.json` or `yarn.lock` pinning `axios@1.14.1`, `axios@0.30.4`, or `plain-crypto-js` at any version is evidence of exposure. The directory `node_modules/plain-crypto-js/` should not exist in any legitimate project.
3. **Don't trust npm list.** The dropper replaces its own `package.json` with a clean stub after execution. Post-compromise filesystem inspection will show version 4.2.0 (clean) instead of 4.2.1 (malicious). EDR telemetry from the time of install is the only reliable evidence.
4. **Pin dependencies.** Use exact versions in `package.json` and commit lockfiles. The malicious versions were tagged `latest` and `legacy`, so any project with `"axios": "^1.14.0"` or `"axios": "^0.30.0"` in their range would have pulled the backdoored version during the exposure window.
5. **Audit postinstall hooks.** Run `npm query ':attr(scripts, [postinstall])' | jq '.[].name'` against projects to list all packages with postinstall hooks. Consider `--ignore-scripts` for CI builds.
6. **Deploy chokepoint detections.** The renamed PowerShell, node-to-tmp-to-nohup, and osascript-from-tmp Sigma rules from the blog writeup target the technique, not the IOCs.

### MITRE ATT&CK

| ID | Technique | How |
|----|-----------|-----|
| T1195.002 | Supply Chain: Software | Malicious npm package via hijacked maintainer account |
| T1059.007 | Command and Scripting: JavaScript | Obfuscated postinstall dropper (setup.js) |
| T1059.002 | Command and Scripting: AppleScript | macOS dropper via osascript |
| T1059.005 | Command and Scripting: VBScript | Windows dropper via cscript |
| T1059.001 | Command and Scripting: PowerShell | Windows stage-2 via renamed hidden PowerShell |
| T1036.005 | Masquerading: Match Legitimate Name | com.apple.act.mond (macOS), wt.exe (Windows) |
| T1070.004 | Indicator Removal: File Deletion | setup.js self-delete + package.json swap |
| T1571 | Non-Standard Port | C2 on port 8000 |